# Task A -- does a third ensemble member help?

The current best Task A submission blends two models, TF-IDF/SVM and MuRIL, and beat MuRIL
alone. Blending works when members make *different* mistakes, so the natural next question
is whether a third member with a genuinely different inductive bias adds anything.

XLM-R is the candidate. It uses sentencepiece rather than wordpiece, was pretrained on a
different corpus, and on Task B scored 0.5561 against MuRIL's 0.5948. **A weaker member
can still improve a blend** if its errors are decorrelated, which is exactly what this
notebook measures rather than assumes.

| member | why it is in the blend |
|---|---|
| TF-IDF / LinearSVC | character n-grams, robust to the spelling variation in romanized Kannada |
| MuRIL | pretrained on transliterated Indic text, the strongest single model |
| XLM-R | different tokenizer and pretraining corpus, so different failure modes |

## Two XLM-R settings that are not the MuRIL defaults

`--no-demojize`: MuRIL's wordpiece vocabulary has no emoji and maps the whole containing
word to unknown, which is why demojization is on by default. Sentencepiece keeps emoji
natively, so rewriting them is at best neutral.

`--epochs 8`: on Task B, XLM-R needed two full epochs just to escape the trivial solution
with the loss pinned at ln 2. Six may not be enough.

## Runtime

About **6 hours**: minutes for the SVM, 160 for MuRIL, 200 for XLM-R. Then the blend
search costs no GPU at all.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Out-of-fold probabilities for all three members

All on split seed 42, so the three matrices line up row for row. Without that they cannot
be blended honestly.

In [ ]:
import time
t0 = time.time()
BUDGET_H, RESERVE_MIN = 10.5, 20
left = lambda: BUDGET_H * 3600 - (time.time() - t0) - RESERVE_MIN * 60

from sklearn.metrics import f1_score
from hastika.common.preprocessing import clean
df = train.iloc[keep].reset_index(drop=True)
X = np.array([clean(t, demojize=True) for t in df["Comment"]])
y = (df["Label"] == "Hate").astype(int).values
oof_of = lambda tag: np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")
score_of = lambda tag: f1_score(y, oof_of(tag).argmax(1), average="macro")

SVM_TAG = "task_a_blend_svm"
run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_TAG, "--demojize"], log=f"artifacts/logs/{SVM_TAG}.log")

COMMON = ["--folds", "5", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
          "--select", "last", "--reinit-layers", "1", "--seeds", "42"]
ARMS = [("task_a_blend_muril", ["--model", "google/muril-base-cased", "--epochs", "6"], 160),
        ("task_a_blend_xlmr",  ["--model", "xlm-roberta-base", "--epochs", "8",
                                "--no-demojize"], 200)]
members = [SVM_TAG]
for tag, extra, est in ARMS:
    if left() < est * 60:
        print(f"skip {tag}: {left()/60:.0f} min left, needs ~{est}", flush=True)
        continue
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag,
         *COMMON, *extra], log=f"artifacts/logs/{tag}.log")
    members.append(tag)
print("\nmembers with OOF:", members)

## 2. How different are their mistakes?

This is the diagnostic that predicts whether blending can help at all. Two members that
agree on 95% of rows have little to offer each other however good they are individually.
Disagreement rate matters more than the third member's own score.

In [ ]:
P = {m: oof_of(m) for m in members}
for m in members:
    print(f"  {m:22s} OOF macro-F1 {score_of(m):.4f}")
print("\npairwise disagreement on argmax:")
for i, a in enumerate(members):
    for b in members[i + 1:]:
        da = (P[a].argmax(1) != P[b].argmax(1)).mean()
        both = ((P[a].argmax(1) == y) & (P[b].argmax(1) == y)).mean()
        either = ((P[a].argmax(1) == y) | (P[b].argmax(1) == y)).mean()
        print(f"  {a:22s} vs {b:22s} disagree {da:.3f}  both right {both:.3f}  "
              f"either right {either:.3f}")
print("\n'either right' is the ceiling a perfect blend of that pair could reach")

## 3. Nested blend search over all members

Weights are searched on a simplex grid together with the decision threshold, and both are
chosen on an inner split of each fold's training rows. The in-sample number is printed for
shape only; the nested one is the result.

In [ ]:
from sklearn.model_selection import StratifiedKFold
import itertools

def blend(ws, idx):
    return sum(w * P[m][idx, 1] for w, m in zip(ws, members))

STEP = 0.1
GRID_W = [w for w in itertools.product(np.arange(0, 1.0001, STEP), repeat=len(members))
          if abs(sum(w) - 1) < 1e-6]
GRID_T = np.arange(0.34, 0.67, 0.02)
print(f"{len(GRID_W)} weight combinations x {len(GRID_T)} thresholds")

def sc(ws, t, idx):
    return f1_score(y[idx], (blend(ws, idx) > t).astype(int), average="macro")

allidx = np.arange(len(y))
bw, bt = max(((w, t) for w in GRID_W for t in GRID_T), key=lambda p: sc(p[0], p[1], allidx))
print(f"in-sample best: {dict(zip(members, np.round(bw, 2)))} t={bt:.2f} "
      f"-> {sc(bw, bt, allidx):.4f}  (optimistic)")

pred = np.zeros(len(y), dtype=int); picks = []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y):
    itr, iva = next(StratifiedKFold(4, shuffle=True, random_state=7).split(np.zeros(len(tr)), y[tr]))
    inner = tr[iva]
    w, t = max(((w, t) for w in GRID_W for t in GRID_T), key=lambda p: sc(p[0], p[1], inner))
    picks.append((tuple(np.round(w, 2)), round(t, 2)))
    pred[va] = (blend(w, va) > t).astype(int)
nested = f1_score(y, pred, average="macro")
print(f"\nnested blend macro-F1 {nested:.4f}")
for m in members:
    print(f"  vs {m:22s} {nested - score_of(m):+.4f}")
print("per-fold picks:", picks)

## 4. Preserve outputs

The `oof_probs.npy` files are the durable product here. With all three stored, any future
blend idea is a few seconds of CPU rather than six hours of GPU.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_blend_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag in members:
    for name in ["oof_probs.npy", "test_probs.npy"]:
        p = pathlib.Path("artifacts/runs") / tag / name
        if p.exists():
            shutil.copy2(p, OUT / f"{tag}_{name}")
    log = pathlib.Path(f"artifacts/logs/{tag}.log")
    if log.exists():
        shutil.copy2(log, OUT / log.name)
json.dump({"members": members, "oof": {m: score_of(m) for m in members},
           "nested": nested, "picks": [[list(w), t] for w, t in picks]},
          open(OUT / "blend_summary.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 5. What to do with the result

Record the nested blend score and every member's own out-of-fold score.

Adopt the third member only if the nested blend beats the best two-member blend by more
than about a point. A weight search over three members has more freedom to fit noise than
one over two, which is exactly why the estimate here is nested.

No submission is produced. If the three-way blend wins, the full-data build is a separate
run using the weights recorded in `blend_summary.json`.